In [1]:
# =============================================================================
# CASE STUDY: AIC AND BIC FOR MODEL SELECTION
# Disease progression in diabetic patients — sklearn.datasets.load_diabetes
# =============================================================================
# Variables:
#   Y   : Disease progression one year after baseline  [response]
#   X1  : age     — age in years
#   X2  : sex     — biological sex
#   X3  : bmi     — body mass index
#   X4  : bp      — average blood pressure
#   X5  : s1 (tc) — total serum cholesterol
#   X6  : s2 (ldl)— low-density lipoproteins
#   X7  : s3 (hdl)— high-density lipoproteins
#   X8  : s4 (tch)— total cholesterol / HDL ratio
#   X9  : s5 (ltg)— log of serum triglycerides
#   X10 : s6 (glu)— blood sugar level
# Note: all features are mean-centered and scaled in the original dataset.
# =============================================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
from itertools import combinations
from sklearn.datasets import load_diabetes

# =============================================================================
# 1. LOAD DATASET
# =============================================================================

data    = load_diabetes()
df      = pd.DataFrame(data.data, columns=data.feature_names)
df["y"] = data.target

FEATURES = data.feature_names
print("=" * 58)
print("DIABETES DATASET — first rows")
print("=" * 58)
print(df.head(5).round(3).to_string(index=False))
print(f"\nn = {len(df)} observations  |  p = {len(FEATURES)} candidate predictors")
print(f"Target (y) — mean: {df['y'].mean():.1f}  std: {df['y'].std():.1f}")

# =============================================================================
# 2. AIC AND BIC — REMINDER
# =============================================================================
#
#   p   = number of parameters (k predictors + 1 intercept)
#   SSR = sum of squared residuals
#
#   AIC = n · log(SSR / n) + 2 · p
#   BIC = n · log(SSR / n) + log(n) · p
#
#   Here log(n) = log(442) ≈ 6.09  >>  2
#   → BIC penalizes complexity much more heavily than AIC.
#
# =============================================================================

def fit_ols(features):
    X = sm.add_constant(df[list(features)])
    return sm.OLS(df["y"], X).fit()

# =============================================================================
# 3. FORWARD SELECTION  —  criterion: AIC
# =============================================================================

print("\n" + "=" * 58)
print("FORWARD SELECTION  —  criterion: AIC")
print("=" * 58)
print(f"\n{'Step':<5} {'Added':<8} {'k':>3} {'AIC':>9} {'BIC':>9}")
print("-" * 36)

selected = []
remaining = list(FEATURES)

while remaining:
    best_aic, best_var = np.inf, None
    for var in remaining:
        res = fit_ols(selected + [var])
        if res.aic < best_aic:
            best_aic, best_var = res.aic, var
    if selected:
        if best_aic >= fit_ols(selected).aic:
            break
    selected.append(best_var)
    remaining.remove(best_var)
    res = fit_ols(selected)
    print(f"{len(selected):<5} {best_var:<8} {len(selected)+1:>3} {res.aic:>9.2f} {res.bic:>9.2f}")

selected_fw = selected.copy()
print(f"\nForward (AIC) model: {selected_fw}")

# =============================================================================
# 4. BACKWARD ELIMINATION  —  criterion: BIC
# =============================================================================

print("\n" + "=" * 58)
print("BACKWARD ELIMINATION  —  criterion: BIC")
print("=" * 58)
print(f"\n{'Step':<5} {'Removed':<8} {'k':>3} {'AIC':>9} {'BIC':>9}")
print("-" * 36)

selected = list(FEATURES)

while len(selected) > 1:
    current_bic = fit_ols(selected).bic
    best_bic, worst_var = np.inf, None
    for var in selected:
        candidate = [v for v in selected if v != var]
        res = fit_ols(candidate)
        if res.bic < best_bic:
            best_bic, worst_var = res.bic, var
    if best_bic >= current_bic:
        break
    selected.remove(worst_var)
    res = fit_ols(selected)
    print(f"{len(selected):<5} {worst_var:<8} {len(selected)+1:>3} {res.aic:>9.2f} {res.bic:>9.2f}")

selected_bw = selected.copy()
print(f"\nBackward (BIC) model: {selected_bw}")

# =============================================================================
# 5. COMPARISON
# =============================================================================

print("\n" + "=" * 58)
print("MODEL COMPARISON")
print("=" * 58)
print(f"\n{'Model':<22} {'Features':<32} {'k':>3} {'AIC':>9} {'BIC':>9}")
print("-" * 68)

for label, feats in [("Full model",       list(FEATURES)),
                     ("Forward  (AIC)",   selected_fw),
                     ("Backward (BIC)",   selected_bw)]:
    res  = fit_ols(feats)
    abbr = " ".join(feats)
    print(f"{label:<22} {abbr:<32} {len(feats)+1:>3} {res.aic:>9.2f} {res.bic:>9.2f}")

in_fw  = set(selected_fw)
in_bw  = set(selected_bw)
dropped = in_fw - in_bw
added   = in_bw - in_fw

print(f"\nForward  kept  : {sorted(in_fw)}")
print(f"Backward kept  : {sorted(in_bw)}")
if dropped:
    print(f"\nBIC dropped vs AIC: {sorted(dropped)}")
    print("→ Those predictors improved AIC but not enough to offset BIC's")
    print(f"  stricter penalty (log({len(df)}) = {np.log(len(df)):.2f} vs 2 per parameter).")
else:
    print("\nBoth criteria selected the same features.")

DIABETES DATASET — first rows
   age    sex    bmi     bp     s1     s2     s3     s4     s5     s6     y
 0.038  0.051  0.062  0.022 -0.044 -0.035 -0.043 -0.003  0.020 -0.018 151.0
-0.002 -0.045 -0.051 -0.026 -0.008 -0.019  0.074 -0.039 -0.068 -0.092  75.0
 0.085  0.051  0.044 -0.006 -0.046 -0.034 -0.032 -0.003  0.003 -0.026 141.0
-0.089 -0.045 -0.012 -0.037  0.012  0.025 -0.036  0.034  0.023 -0.009 206.0
 0.005 -0.045 -0.036  0.022  0.004  0.016  0.008 -0.003 -0.032 -0.047 135.0

n = 442 observations  |  p = 10 candidate predictors
Target (y) — mean: 152.1  std: 77.1

FORWARD SELECTION  —  criterion: AIC

Step  Added      k       AIC       BIC
------------------------------------
1     bmi        2   4912.04   4920.22
2     s5         3   4828.40   4840.67
3     bp         4   4813.23   4829.59
4     s1         5   4804.96   4825.42
5     sex        6   4800.08   4824.63
6     s2         7   4788.60   4817.24

Forward (AIC) model: ['bmi', 's5', 'bp', 's1', 'sex', 's2']

BACKWARD ELIM